## Study Planner AI Agent

- Study Planner Agent creates personalized study schedules based on user input and deadlines.
- Tasks are automatically generated and pushed directly to Jira as issues for easy tracking and execution.
- The Study Planner Agent Extension can book calendar reminders to help you stay on track.

### Install Dependencies

In [ ]:
!pip install Phidata Groq youtube_transcript_api dotenv jira pycountry

# firecrawl langchain openai langchain_openai langchain_community pyngrok

## Stage 1: Setup Initialization

#### [Pre-requisite] Get an API key for your Groq

#### Step 1: Visit the Groq Developer Portal
- Open your browser and go to: https://console.groq.com

#### Step 2: Sign Up or Log In
- If you already have an account, click Log In.
- If you’re new, click Sign Up and follow the prompts to create an account (you may need to verify your email).

#### Step 3: Access the API Section
- Once logged in, you'll land on the Groq Console.
- Navigate to the API Keys section from the sidebar or dashboard.

#### Step 4: Generate a New API Key
- Click the “Create API Key” button.
- Give your key a name (e.g., "test-key").
- Click Create or Generate.

#### Step 5: Copy and Store the Key Securely
- Your API key will be shown only once — copy it immediately and store it in a secured location.
- Never expose your API key in client-side code or public repositories.


### [Optional step if you are using Google Colab]
### Add the API key in the Secret Manager
Step 1: Click on Secrets (Key sign) on the left pane of colab

Step 2: Provide the name as GROQ_API_KEY and Value as API Key copied in Step 5 of 1.1

Step 3: Toggle "ON" the notebook access.

## Stage 2: Setup Jira Account

#### [Pre-requisite] Get API Key for Jira

#### Step 1: Sign up/Log in to the Jira account
- Sign up on the website - https://www.atlassian.com/software/jira using your email address, Google, or Microsoft account.

#### Step 2: How to get Jira API Key
- Go to https://id.atlassian.com/manage-profile/security/api-tokens page
- Click on Create Classic API Token.
- Give a name like "Test API"
- Copy and store the key securely.

#### Step 3: Jira Server URL
- The Jira Server URL is your Atlassian site URL, usually ending in .atlassian.net.

#### Step 4: Jira Username
- We need to set JIRA_USERNAME environment variable with the email address that is used to sign up for the Jira account.

#### [Optional if you are using Google Colab]
#### Step 5: Add the following to the Secret Manager
- Enter JIRA_API_KEY as the name and the <API KEY> as value
- Enter JIRA_SERVER_URL as the name and add <Server URL> (like https://xxx.atlassian.net) as value
- Enter JIRA_USERNAME as the name and add <EMAIL ID> used to sign up as the value.

### 2.2. Create a project
- In the project templates pane, choose Software Development, then on the main window choose Kanban
- Select a team managed project
- On the "Add project details" page, set the name as "Test Project", set the key to "TES" (This is important) and set the access to Open.

## Stage 3: Setup Cal.com Account

#### [Pre-requisite] Get an API key for Cal.com 

#### Step 1: Create an account on Cal.com
- Setup an account on Cal.com if it does not exist
- In the availability on the left pane, keep all slots open (note if the slot is not open, we cannot create a meeting invite).

#### Step 2: Get API Key
- Once logged in, go to https://app.cal.com/settings/developer/api-keys
- Click “+ Add”
- Give it a name - "Test AI Agent"
- Copy and store the key securely.

#### Step 3: Get Event type id
- Run on curl.exe -H "Authorization: Bearer <CALCOM_API_KEY>" https://api.cal.com/v2/event-types on powershell (windows). If you are using Mac, use "curl" instead of "curl.exe".
- Event type id is the value of id in the eventTypes element

#### [Optional if you are using Google Colab]
#### Step 4: Add the following to the Secret Manager
- Enter CALCOM_API_KEY as the name and the <API KEY> as value
- Enter CALCOM_EVENT_TYPE_ID as the name and add the id from Step 3 above
- Enter EMAIL_ID as the name and add <EMAIL ID> to which the calendar invitations need to be sent.

In [16]:
import os

os.environ["GROQ_API_KEY"] = "enter-your-groq-api-key"
os.environ["JIRA_SERVER_URL"] = "enter-your-jira-server-url"
os.environ["JIRA_USERNAME"] = "enter-your-jira-username"
os.environ["JIRA_TOKEN"] = "enter-your-jira-token"
os.environ["CALCOM_API_KEY"] = "enter-your-calcom-api-key"
os.environ["CALCOM_EVENT_TYPE_ID"] = "enter-your-calcom-event-type-id"

In [ ]:
import os
from phi.agent import Agent
from phi.model.groq import Groq
from phi.tools.jira_tools import JiraTools
from phi.tools.calcom import CalCom
from datetime import datetime

email_id = "enter-the-email-id"
project_id = "enetr-your-project-id"

# Jira Agent
jira_agent = Agent(
    name="Jira Study Planner",
    model=Groq(id="enter-your-model-id"),
    tools=[JiraTools()],
    markdown=True,
    instructions=[
        f"Today is {datetime.now()}.",
        "When creating Jira issues always pass project_key, summary, description, and issuetype='Task'.",
    ],
)

# CalCom Agent
cal_agent = Agent(
    name="Calendar Booking Agent",
    model=Groq(id="enter-your-model-id"),
    tools=[CalCom()],
    markdown=True,
    instructions=[
        f"Today is {datetime.now()}.",
        "Create calendar bookings as requested.",
    ],
)

# Create Jira tasks
jira_agent.print_response(
    f"""Create 14 Jira tasks in project {project_id} for a 2-week machine learning study plan.
    One task per day. Each task must have:
    - project_key: TES
    - summary: 'Day X: Topic Name'
    - description: brief 2-line study plan for that day
    - issuetype: Task
    Create all 14 tasks now directly.""",
    stream=True
)

# Create CalCom bookings
cal_agent.print_response(
    f"""Create 14 daily bookings with {email_id} for the next 14 days at 9pm IST.
    Each booking subject should match the ML study topic for that day.
    Create all bookings now directly.""",
    stream=True
)

## Conclusion

This project built an AI-powered Study Planner Agent using the Phidata framework and Groq's LLM, integrated with Jira for task creation and Cal.com for calendar bookings — all triggered by a single natural language prompt. Running in a Jupyter Notebook, the agent automatically generated a personalized 14-day machine learning study plan without any manual intervention. Several challenges were resolved along the way, including fixing environment variable mismatches, handling decommissioned Groq models, and splitting the agent into two focused agents to avoid tool-calling failures. The project demonstrated how agentic AI workflows can connect natural language understanding with real-world productivity tools, turning a simple user request into a fully automated, actionable study schedule.

## AUTHOR
Jessica Lenifer R